# `notebook_2_model` — инференс и submission

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from sygnal_clustering.config import ARTIFACTS_DIR, DATA_PATH, SUBMISSION_PATH
from sygnal_clustering.data import load_waveforms
from sygnal_clustering.pipeline import SygnalClusteringPipeline

pipe = SygnalClusteringPipeline.load(ARTIFACTS_DIR / "pipeline.joblib")
X = load_waveforms(DATA_PATH)
labels = pipe.fit_predict(X)
sub_path = pipe.save_submission(SUBMISSION_PATH)
inf_metrics = pipe.metrics()
sub_df = pd.read_csv(sub_path)
infer_result = {
    "n_rows": len(sub_df),
    "clusters": sorted(sub_df["cluster"].unique().tolist()),
    "metrics": inf_metrics,
    "path": str(sub_path),
}
print(infer_result)
display(sub_df.head())

## Kaggle — таблица лидеров (первая отправка)

Скриншот лидерборда сохранён в репозитории: `Разработка/kaggle_leaderboard_first_submission.png` (соревнование «Классификация типов сигналов»).

In [ ]:
from IPython.display import Image, display

LEADERBOARD_IMG = ROOT / "Разработка" / "kaggle_leaderboard_first_submission.png"
kaggle_leaderboard = {
    "competition": "Классификация типов сигналов",
    "image_path": str(LEADERBOARD_IMG),
    "image_exists": LEADERBOARD_IMG.exists(),
    "rank": 33,
    "score": 0.36568,
    "submissions": 1,
}
print(kaggle_leaderboard)
if kaggle_leaderboard["image_exists"]:
    display(Image(filename=str(LEADERBOARD_IMG)))
else:
    print("Файл скриншота не найден:", LEADERBOARD_IMG)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f'''### Kaggle — ML-архитектор
Первая отправка: место **#{kaggle_leaderboard["rank"]}**, accuracy **{kaggle_leaderboard["score"]:.5f}** (цель ТЗ ≥ 0.84, целевой порог > 0.85). Внутренний silhouette **{infer_result["metrics"]["silhouette"]:.4f}** не коррелирует с лидербордом — нужна калибровка меток/порогов под метрику Kaggle.

### Kaggle — физик
Score **{kaggle_leaderboard["score"]:.5f}** указывает на неверное сопоставление кластеров 0/1 с γ и нейтронами или избыточный кластер 2; требуется итерация по PSD-порогам и перекодировке после получения обратной связи с лидерборда.'''))

In [ ]:
from IPython.display import Markdown, display

display(Markdown('''### ML-архитектор
Инференс: **{infer_result[n_rows]}** строк, кластеры **{infer_result[clusters]}**. Silhouette **{infer_result[metrics][silhouette]:.4f}**. Файл: `{infer_result[path]}`.

### Физик
Распределение кластеров воспроизводит обучение; готово к загрузке на Kaggle.'''))